# Phase 7: Probability and Statistical Analysis
## Multi-Objective Recommender System

**Objective:** Transition from visual observations (EDA) to mathematical proofs. We will quantify the characteristics of our dataset to justify the architectural decisions of the recommender system.

**Key Statistical Targets:**
1. **Dataset Sparsity:** Quantify the 'emptiness' of the user-item matrix.
2. **Correlation Analysis:** Mathematically prove the relationship between price and rating.
3. **Distributional Skewness:** Analyze the bias in user ratings.
4. **Gini Coefficient:** Quantify the popularity concentration (The Long Tail proof).

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

# Use the absolute path that we verified works in your environment
data_processed_path = Path(r'C:\Users\DrX DIPAK\projects\multi-objective-recommender\data\processed')

try:
    ratings = pd.read_csv(data_processed_path / 'ratings_processed.csv')
    metadata = pd.read_csv(data_processed_path / 'metadata_processed.csv')
    user_summary = pd.read_csv(data_processed_path / 'user_summary.csv')
    item_summary = pd.read_csv(data_processed_path / 'item_summary.csv')
    print("✅ Statistical datasets loaded successfully.")
except Exception as e:
    print(f"❌ Loading Error: {e}")

### 1. Dataset Sparsity Analysis
**Business Meaning:** Sparsity measures what percentage of the user-item matrix is empty. If sparsity is >99%, a simple collaborative filtering model will fail (The Cold Start Problem), necessitating a multi-objective approach that uses metadata (Content-Based filtering).

In [ ]:
# Calculate total possible interactions
n_users = ratings['user_id'].nunique()
n_items = ratings['item_id'].nunique()
total_possible = n_users * n_items
total_actual = len(ratings)

sparsity = (1 - (total_actual / total_possible)) * 100

print(f"Total Users: {n_users}")
print(f"Total Items: {n_items}")
print(f"Total Possible Interactions: {total_possible}")
print(f"Actual Interactions: {total_actual}")
print(f"\nDataset Sparsity: {sparsity:.4f}%")

if sparsity > 95:
    print("\nInterpretation: HIGH SPARSITY. This confirms we cannot rely on User-Item interactions alone. We MUST use item metadata to fill the gaps.")
else:
    print("\nInterpretation: MODERATE SPARSITY.")

### 2. Correlation Analysis (Price vs Rating)
**Business Meaning:** We use Pearson (linear) and Spearman (rank) correlation to see if higher prices lead to higher (or lower) ratings. This tells us if 'Luxury' items are perceived as better quality.

In [ ]:
# Merge ratings with metadata to get prices
df_corr = ratings.merge(metadata[['item_id', 'price']], on='item_id', how='left').dropna()

pearson_corr, p_val_p = stats.pearsonr(df_corr['price'], df_corr['rating'])
spearman_corr, p_val_s = stats.spearmanr(df_corr['price'], df_corr['rating'])

print(f"Pearson Correlation: {pearson_corr:.4f} (p-value: {p_val_p:.4f})")
print(f"Spearman Correlation: {spearman_corr:.4f} (p-value: {p_val_s:.4f})")

if p_val_p < 0.05:
    print("\nInterpretation: The correlation is Statistically Significant.")
else:
    print("\nInterpretation: No statistically significant linear relationship found between Price and Rating.")

### 3. Rating Distribution (Skewness & Kurtosis)
**Business Meaning:** Skewness tells us if ratings are biased. If skewness is negative, users are 'too generous' (ratings cluster at 5). This informs how we normalize ratings for the model.

In [ ]:
skewness = stats.skew(ratings['rating'])
kurtosis = stats.kurtosis(ratings['rating'])

print(f"Rating Skewness: {skewness:.4f}")
print(f"Rating Kurtosis: {kurtosis:.4f}")

if skewness < 0:
    print("\nInterpretation: Negative Skew. The distribution is biased toward higher ratings (generous users).")
elif skewness > 0:
    print("\nInterpretation: Positive Skew. The distribution is biased toward lower ratings (harsh users).")
else:
    print("\nInterpretation: Symmetric distribution.")

### 4. The Gini Coefficient (Popularity Concentration)
**Business Meaning:** The Gini coefficient is used in economics to measure inequality. In RecSys, it measures 'Popularity Bias'. A Gini close to 1.0 means a tiny fraction of items get all the attention, proving the Long Tail effect.

In [ ]:
def calculate_gini(array):
    """Calculate the Gini coefficient of a numpy array."""
    array = np.array(array, dtype=np.float64)
    if np.amin(array) < 0: 
        array -= np.amin(array) # Gini requires non-negative values
    array += 0.0000001 # Avoid division by zero
    array = np.sort(array)
    index = np.arange(1, array.shape[0] + 1)
    n = array.shape[0]
    return ((np.sum((2 * index - n  - 1) * array)) / (n * np.sum(array)))

popularity = item_summary['total_interactions'].values
gini_index = calculate_gini(popularity)

print(f"Popularity Gini Coefficient: {gini_index:.4f}")

if gini_index > 0.6:
    print("\nInterpretation: HIGH CONCENTRATION. The 'Long Tail' is severe. This mathematically justifies the need for a Novelty objective to surface under-rated items.")
else:
    print("\nInterpretation: Low to Moderate concentration.")